# Web Scraping Lecture Examples

## BeautifulSoup and requests

We would like to obtain a dataset of all the football world cup finals (who won, against who, where was the final held...). Wikipedia has a pretty complete table in the following page: https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals

![](assets/fifa-table.png)

We are going to scrape the contents of the table and save it as our own dataset in csv.

### Installation

First make sure you have both libraries installed. If you don't, install them with:

```bash
pip install requests
pip install beautifulsoup4
```

### Getting the website content

The first part consists of getting a copy of the contents of the page we want to scrape in our computer. **It is very important to design our code in a way that the contents are only downloaded when needed, and only what we need to make sure we don't overload te server**. For that, we will define a function that checks if we already have a copy of the website and if not makes a request to get it and store it in a cache. There should be as well a flag that indicates if we want to force the download of the web content (for example after the next worldcup, to make sure to get the latest data).

In [ ]:
# Lets make our imports
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals"

In [ ]:
CACHE_PATH = "./cache/"

def get_page_content(url, update_cache=False):
    # Extract the filename from the URL
    filename = url.split("/")[-1]
    
    # Create the cache directory if it doesn't exist
    os.makedirs(CACHE_PATH, exist_ok=True)
    
    # Check if the file already exists in the cache
    file_path = os.path.join(CACHE_PATH, filename)
    
    if not os.path.exists(file_path) or update_cache:
        # File doesn't exist in the cache or update_cache is True, so download it
        response = requests.get(url, verify=False, headers={'User-Agent': 'Magic Browser'})
        
        if response.status_code == 200:
            # Save the content to the cache
            with open(file_path, 'wb') as file:
                file.write(response.content)
            print(f"Downloaded and cached: {url}")
            return response.content.decode('utf-8')
        else:
            print(f"Failed to download: {url}")
            return None
    else:
        # File already exists in the cache, so read and return its content
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        print(f"Loaded from cache: {url}")
        return content


content = get_page_content("https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals")


See how the contents are saved to a cache the first time the function is called and is only downloaded again if the update flag is true.

### Extracting the information

Now that we have the contents in our system we can use Beautiful Soup to extract the table from the HTML. Let's try to obtain the table.

In [ ]:
content

In [ ]:
soup = BeautifulSoup(content, "html.parser")
table = soup.find("table")
print(table)

This is no the table we expected. Indeed it has the contents of the **first** `<table>` element of the page, which is not the table we want.

![](./assets/first-table.png)

In order to get our table, we need to check the structure of the HTML document and try to find a way to uniquely identify the table element that corresponds to our table and query it with Beautiful Soup. For that the web inspector is very useful:

![](./assets/fifa-inspector.png)

We can see that the table is an element of type `<table>` and that it has a class name `sortable plainrowheaders wikitable jquery-tablesorter`. Let's try to query the elements of type table with that class name.

In [ ]:
soup = BeautifulSoup(content, "html.parser")
table = soup.find("table", {"class": "sortable plainrowheaders wikitable jquery-tablesorter"})
print(table)

It appears that there are no elements with the attributes specified in the query. By looking closely to the class name, we see the last part, `jquery-tablesorter` is something that has to do with `jquery` which is a JS framework. This means that the class name was probably modified by the JS thread, so we should look in our cache file what the class name of the table is there, which results to be `sortable plainrowheaders wikitable`. Let's try with this now:

In [ ]:
soup = BeautifulSoup(content, "html.parser")
table = soup.find("table", {"class": "sortable plainrowheaders wikitable"})
print(table)

### Putting it all together

We are going to create a complete function that dowloads the content if needed, extracts the data from the HTML document and returns the data as a dataframe, ready for cleaning/processing. It will write the data to a CSV as well so that the processing with Beautiful Soup does not need to be redone if the CSV is present. Instead of parsing the data manually, we are going to use pandas to do this (**make sure to have the lxml dependency installed -> `pip install lxml`**). Let's have a look at what this could look like:

In [ ]:
def extract_and_save_table(content, output_path):
    # Get the table contents from the HTML
    soup = BeautifulSoup(content, "html.parser")
    table = soup.find("table", {"class": "sortable plainrowheaders wikitable"})
    # Parse the table into a pandas dataframe, io.StringIO is required by Pandas (see https://pandas.pydata.org/docs/reference/api/pandas.read_html.html)
    table_dfs = pd.read_html(io.StringIO(str(table)))
    # We only extracted one table, so get the first one and write the result to the CSV
    table_df = table_dfs[0]
    table_df.to_csv(output_path, index=False)
    # Return the datagrame
    return table_df

def get_fifa_winners(update=False):
    CSV_CACHE_PATH = os.path.join(CACHE_PATH,"fifa.csv")
    # If the CSV exists, parse and return that
    if os.path.exists(CSV_CACHE_PATH) and not update:
        print("Getting table from cached CSV")
        table_df = pd.read_csv(CSV_CACHE_PATH)
        return table_df
    else:
        print("Getting table from HTML")
        page_content = get_page_content("https://en.wikipedia.org/wiki/List_of_FIFA_World_Cup_finals", update_cache=update)
        table_df = extract_and_save_table(page_content, CSV_CACHE_PATH)
        return table_df

table_df = get_fifa_winners()
table_df.head()


This simple example could have been done using only the `pandas.read_html` function. Indeed, pandas is able to retrieve a URL and uses beautiful soup under the hood to get the tables of a document from selectors. Nevertheless, here we have decided to use directly beautiful soup for several reasons:
- it allows for more control over if we want to download content or use a local cache
- it works for a more general case where the data is not in a table

### Extracting another table from the document

We have successfully extracted the principal table from the document, let's now try to extract this table:

![](./assets/confederation.png)

By checking the web inspector, we can see that it corresponds to a table element with the `wikitable plainrowheaders sortable jquery-tablesorter` class name. We can see it has the same `jquery` suffix as before, so we can go ahead and try to query the tables with the `wikitable plainrowheaders sortable` class name:

In [ ]:
table = soup.find("table", {"class": "wikitable plainrowheaders sortable"})
print(table)

This table is not the one we were hoping for, as it's the one that is just above in the HTML document:

![](./assets/nation.png)

Indeed, by checking again the web inspector, we can see the two tables have the same class name:

![](./assets/classnames.png)

We will need to use a different method in Beautiful Soup to be able to extract the table we want. We use the `findAll` method to get all the tables matching our query and then look for something unique in that table. Here we know the tables have different captions so we will be using that, but more generally this should be adapted to look for what defined the uniqueness of the elements that contains the data we are trying to extract.

In [ ]:
# Use find all to get all the tables matching our query
tables = soup.findAll("table", {"class": "wikitable plainrowheaders sortable"})
soup.find_all("table", {"class": "wikitable plainrowheaders sortable"})
# Initialize table to None
confederation_table = None
# Iterate through the tables to obtain the one that has the caption that we want
for table in tables:
    # First get the table's caption    
    caption = table.find("caption")
    # Check if the caption 
    if caption and "Results by confederation" in caption.text:
        caption_confederation = caption
        confederation_table = table

print(confederation_table)

## Example Salaries

![image.png](attachment:image.png)

In [ ]:
response = requests.get("https://www.nexford.edu/insights/highest-paying-jobs-in-the-world", verify=False)
soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
import re

### Solución 1

In [ ]:
def generate_database(soup: BeautifulSoup()) -> pd.DataFrame:
    pattern_job = re.compile(r'(\d+\.) (([A-Za-z]+\s?)+)')

    jobs = []
    
    for element in soup.find_all('h3'):
        match = re.search(pattern_job, element.text)
        if match != None:
            jobs.append(match.group(2))
    
    pattern_salary = re.compile(r'\$\d+\,\d+')
    
    salaries = []
    
    i = 0
    for element in soup.find_all('a'):
        match = re.search(pattern_salary, element.text)
        if i >= len(jobs):
            break
        if match != None:
            salaries.append(match.group(0))
            i+=1
    
    best_jobs = pd.DataFrame(data=zip(jobs, salaries), columns=['job', 'salary_average'])
    
    best_jobs['job'] = best_jobs['job'].str.lower()

    return best_jobs

In [ ]:
df = generate_database(soup)

In [ ]:
df

### Solución 2

In [ ]:
pattern_h3 = re.compile(r"[0-9]{1,2}\. ([\w\s]+\w)")
pattern_a = re.compile(r"\$[0-9]+\,?[0-9]*")
def generate_database(soup: BeautifulSoup) -> pd.DataFrame:
    h3_elements = soup.find_all("h3")

    jobs = []
    for h3_element in h3_elements:
        match_obj = pattern_h3.match(h3_element.text)
        if match_obj:
            jobs.append(match_obj.group(1).lower())

    salaries = []
    a_elements = soup.find_all("a")
    for a_element in a_elements:
        match_obj = pattern_a.match(a_element.text)
        if match_obj:
            salaries.append(a_element.text)

    df = pd.DataFrame()
    df["jobs"] = jobs
    df["salaries"] = salaries[0:len(jobs)]
    return df

df = generate_database(soup)

In [ ]:
df

## Scrapy

Follow the tutorial in the [Scrapy documentation](https://docs.scrapy.org/en/latest/intro/tutorial.html)

## Selenium

We are going to to extract [the standings data from the NBA official website](https://www.nba.com/standings).
Let's try to get it with a request to check what the data looks like:

In [ ]:
standings_html = requests.get("https://www.nba.com/standings")
with open("standings_request_result.html","w+") as fp:
    fp.write(standings_html.text())

### Installation

Make sure to follow [the instructions](https://selenium-python.readthedocs.io/installation.html) in the documentation to install Selenium and the driver needed for the browser (we will use the Google Chrome one).
Steps for mac (should be similar for other OS):
- Download the right driver for your platform from the [documentation](https://selenium-python.readthedocs.io/installation.html#drivers)
- unzip the downloaded file and move it to somewhere in your path (for example `/usr/local/bin`). You'll probbaly need to use sudo for this
- ensure the driver is executable by doing `sudo chmod +x /usr/local/bin/chromedriver` (modify the path to match where your driver is installed/named)
- install selenium using pip

### Using selenium

Let's first test if the installation worked. The code below should open a chrome window and navigate to the NBA standings page, print the title of the page and save the html contents of the page to the `standings_selenium_result.html` file. By opening that file in the browser we can see that now the table we are interested in is at the bottom of the HTML document. 

In [ ]:
from selenium import webdriver
import time

driver = webdriver.Chrome()
driver.get("https://www.nba.com/standings")
time.sleep(5)
print(driver.title)
with open("standings_selenium_result.html","w+") as fp:
    fp.write(driver.page_source)
driver.quit()

Let's use Pandas now to extract the table into a dataframe and save the data to a CSV. By looking at the CSV there are two tables, one for the eastern and another one for the western conference. Since there are no more tables in the document we can directly use the `read_html` function to read the tables and save them as CSV.

In [ ]:
import pandas as pd
dfs = pd.read_html("./standings_selenium_result.html")
dfs[0].to_csv("eastern_standings.csv", index=False)
dfs[1].to_csv("western_standings.csv", index=False)